# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

01）Counting the number of lines in .csv files using command: "wc -1" and output to using ">" in Task 1.1; <br>
02）Defining a function and its returns in the profile_table function of Task 1.2; <br>
04）Use Jupyter throughout the notebook and pandas  to create dataframes (.pd.dataframe) in 2.1, 3.1, 3.2, 3.3; <br>
05）<br>
06）<br>

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.

def profile_table(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": table.duplicated().sum()
    }


results = []

for name in manifest["files"]:
    table = raw[Path(name).stem]
    results.append(profile_table(name, table))


table_profile = pd.DataFrame(results)

display(table_profile)

table_profile.to_csv(PROFILE_PATH, index=False)

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [4]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
north = raw["visits_north"].copy(deep=True)

south = raw["visits_south"].copy(deep=True)
south = south.rename(columns={"wt_kg": "weight_kg"})

north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"

visits_raw = pd.concat(
    [north, south],
    ignore_index=True
)

### 2.3 Audit the raw tables

In [5]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and count, and save it to AUDIT_PATH.

patients = raw["patients"]

sites = raw["sites"]

birth_year_num = pd.to_numeric(
    patients["birth_year"],
    errors="coerce"
)

sbp_num = pd.to_numeric(
    visits_raw["sbp_mmhg"],
    errors="coerce"
)


# 1. gender values that are not a code
gender_stripped = patients["gender"].fillna("").str.strip()

patients_gender_not_code = (
    (gender_stripped != "") &
    (~patients["gender"].isin(["F", "M", "X"]))
).sum()

# 2. empty gender
patients_empty_gender = (
    gender_stripped == ""
).sum()

# 3. birth_year sentinel 9999
patients_birth_year_9999 = (
    birth_year_num == 9999
).sum()

# 4. site codes needing normalization
normalized_patient_site = (
    patients["site_code"]
    .str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace("-", "", regex=False)
)

patients_site_need_normalization = (
    patients["site_code"] != normalized_patient_site
).sum()

# 5. rows with a repeated site_code
sites_repeated_code = (
    sites.duplicated(
        subset=["site_code"],
        keep=False
    )
).sum()

# 6. visits with labels that are not baseline or week12
visits_bad_label = (
    ~visits_raw["visit"].isin(["baseline", "week12"])
).sum()

# 7. sbp sentinel 999
visits_sbp_999 = (
    sbp_num == 999
).sum()

# 8. sbp outside 70 through 250
visits_sbp_outside = (
    (sbp_num != 999) &
    (
        (sbp_num < 70) |
        (sbp_num > 250)
    )
).sum()


# 9. date parse failures
date_text = visits_raw["visit_date"].astype("string")

valid_date_format = date_text.str.fullmatch(
    r"\d{4}-\d{2}-\d{2}",
    na=False
)

valid_calendar_date = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
).notna()

visits_date_failures = (
    ~(valid_date_format & valid_calendar_date)
).sum()


# 10. rows in repeated patient_id and visit pairs
visit_for_pair = (
    visits_raw["visit"]
    .str.strip()
    .str.lower()
    .replace({
        "wk12": "week12"
    })
)

pair_table = visits_raw.assign(
    visit_for_pair=visit_for_pair
)

visits_repeated_pairs = (
    pair_table.duplicated(
        subset=["patient_id", "visit_for_pair"],
        keep=False
    )
).sum()

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs"
    ],
    "count": [
        patients_gender_not_code,
        patients_empty_gender,
        patients_birth_year_9999,
        patients_site_need_normalization,
        sites_repeated_code,
        visits_bad_label,
        visits_sbp_999,
        visits_sbp_outside,
        visits_date_failures,
        visits_repeated_pairs
    ]
})

display(issue_audit)

issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,322


## Task 3: Clean

### 3.1 Clean the patients

In [6]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.

display(raw["patients"]["gender"].value_counts(dropna=False))

patients_clean = raw["patients"].copy(deep=True)

# 1. remove exact duplicate rows, keeping the first
patients_clean = patients_clean.drop_duplicates()

# 2. clean gender
gender_map = {
    "female": "F",
    "woman": "F",
    "f": "F",

    "male": "M",
    "man": "M",
    "m": "M",

    "nonbinary": "X",
    "nb": "X",
    "x": "X"
}

patients_clean["gender"] = (
    patients_clean["gender"]
    .str.strip()
    .str.lower()
    .str.replace("-", "", regex=False)
    .map(gender_map)
    .astype("category")
)

# 3. birth_year: 9999 -> missing, dtype Int64
patients_clean["birth_year"] = (
    pd.to_numeric(patients_clean["birth_year"], errors="coerce")
    .replace(9999, pd.NA)
    .astype("Int64")
)

# 4. normalize site_code
display(patients_clean["site_code"].value_counts(dropna=False))

patients_clean["site_code"] = (
    patients_clean["site_code"]
    .str.strip()
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

# 5. reset index
patients_clean = patients_clean.reset_index(drop=True)

summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04"
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        (patients_clean["site_code"] == "N01").sum(),
        (patients_clean["site_code"] == "S02").sum(),
        (patients_clean["site_code"] == "E04").sum()
    ]
})

display(summary)

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

site_code
S02      14771
N01      12895
E04        526
S-02       260
s02        251
n01        224
N 01       222
 S 02      221
 N01       217
S02        203
N-01       181
e04         16
E-04        13
Name: count, dtype: int64[pyarrow]

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.


display(patients_clean["gender"].cat.categories)

summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')

### 3.2 Clean the visits

In [8]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.

visits_clean = visits_raw.copy(deep=True)

# 1: remove exact duplicate rows
visits_clean = visits_clean.drop_duplicates()

# 2: clean visit labels
display(visits_clean["visit"].value_counts(dropna=False))

visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "wk12": "week12",
}

visits_clean["visit"] = (
    visits_clean["visit"]
    .str.strip()
    .str.lower()
    .str.replace(" ", "", regex=False)
    .map(visit_map)
)

# 3: clean sbp_mmhg
visits_clean["sbp_mmhg"] = pd.to_numeric(
    visits_clean["sbp_mmhg"],
    errors="coerce"
)

visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].where(
    visits_clean["sbp_mmhg"].between(70, 250)
)

visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].astype("Int64")

# 4: clean weight_kg
visits_clean["weight_kg"] = pd.to_numeric(
    visits_clean["weight_kg"],
    errors="coerce"
)

# 5: keep visit_date only when exact YYYY-MM-DD and valid calendar date
date_text = visits_clean["visit_date"].astype("string")

exact_date = date_text.str.fullmatch(
    r"\d{4}-\d{2}-\d{2}",
    na=False
)

parsed_date = pd.to_datetime(
    date_text,
    format="%Y-%m-%d",
    errors="coerce"
)

visits_clean["visit_date"] = parsed_date.where(exact_date)

display(visits_clean["visit"].value_counts(dropna=False))
display(visits_clean.describe())

visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

visit
baseline    29676
week12      27625
Name: count, dtype: int64

,visit_date,sbp_mmhg,weight_kg
count,57132,56845.0,56472.0
mean,2026-03-26 07:29:13.497164,140.177236,88.732899
min,2026-01-05 00:00:00,70.0,40.3
25%,2026-02-12 00:00:00,131.0,76.4
50%,2026-03-24 00:00:00,140.0,87.6
75%,2026-05-07 00:00:00,149.0,100.0
max,2026-07-06 00:00:00,250.0,170.4
std,NaN,12.858826,17.271095


In [9]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.

# show the repeated patient_id and visit pairs
repeated_pairs = visits_clean[
    visits_clean.duplicated(
        subset=["patient_id", "visit"],
        keep=False
    )
].sort_values(
    ["patient_id", "visit", "visit_time"]
)

display(repeated_pairs)

# Keep the later reading within each patient_id + visit pair
visits_clean = (
    visits_clean
    .sort_values("visit_time")
    .drop_duplicates(
        subset=["patient_id", "visit"],
        keep="last"
    )
)

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [ ]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).

# sort, renumber, and keep columns in required order
visits_clean = (
    visits_clean
    .sort_values(["patient_id", "visit"])
    .reset_index(drop=True)
)

visits_clean = visits_clean[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "source_file"
    ]
]

# create and save visits summary
visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv"
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum()
    ]
})

display(visits_summary)

visits_summary.to_csv(
    VISITS_SUMMARY_PATH,
    index=False
)


,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [23]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = pd.DataFrame([
    {
        "table": "patients",
        "issue": "exact duplicate rows",
        "decision": "remove exact duplicate rows",
        "reason": "Exact duplicate rows contain no additional information."
    },
    {
        "table": "patients",
        "issue": "gender spellings that are not a code",
        "decision": "Strip and lowercase gender values, map recognized spellings to F, M, or X, and set unrecognized values to missing.",
        "reason": "There are multiple spelling schemes for the same gender categories. Unrecognized and unknown values constitute missing data."
    },
    {
        "table": "patients",
        "issue": "site codes needing normalization",
        "decision": "Uppercase site codes, remove spaces and hyphens.",
        "reason": "There are multiple spelling schemes for the same site. "
    },
    {
        "table": "patients",
        "issue": "birth_year sentinel 9999",
        "decision": "Convert birth_year to nullable Int64 and replace 9999 with a missing value.",
        "reason": "9999 is a sentinel value, it has no meaning."
    },
    {
        "table": "visits",
        "issue": "weight column named wt_kg in visits_south.csv",
        "decision": "Rename wt_kg to weight_kg.",
        "reason": "Standardize column names for stacking."
    },
    {
        "table": "visits",
        "issue": "sbp sentinel 999 and readings outside 70 through 250",
        "decision": "Convert sbp_mmhg to numeric and set 999 and values outside 70 through 250 to missing.",
        "reason": "These values are sentinels or implausible values and therefore are invalid."
    },
    {
        "table": "visits",
        "issue": "impossible and loosely written visit dates",
        "decision": "Keep dates that are only in the YYYY-MM-DD format and parse to real calendar dates, and set all others to missing.",
        "reason": "Remove falsely documented dates."
    },
    {
        "table": "visits",
        "issue": "repeated patient_id and visit pairs",
        "decision": "Sort repeated patient_id and visit pairs by visit_time and keep the last reading.",
        "reason": "The repeated patient_id and visit happen when the same patient took multiple readings on the same day. We keep the later reading to provide one consistent record per patient and visit."
    },
    {
        "table": "sites",
        "issue": "repeated site_code rows",
        "decision": "Keep the repeated site codes.",
        "reason": "The repeated site codes represent records from different clinics despite being in the same region."
    }
])

display(decision_table)

decision_table.to_csv(
    DECISION_LOG_PATH,
    index=False
)

,table,issue,decision,reason
0,patients,exact duplicate rows,remove exact duplicate rows,Exact duplicate rows contain no additional inf...
1,patients,gender spellings that are not a code,"Strip and lowercase gender values, map recogni...",There are multiple spelling schemes for the sa...
2,patients,site codes needing normalization,"Uppercase site codes, remove spaces and hyphens.",There are multiple spelling schemes for the sa...
3,patients,birth_year sentinel 9999,Convert birth_year to nullable Int64 and repla...,"9999 is a sentinel value, it has no meaning."
4,visits,weight column named wt_kg in visits_south.csv,Rename wt_kg to weight_kg.,Standardize column names for stacking.
5,visits,sbp sentinel 999 and readings outside 70 throu...,Convert sbp_mmhg to numeric and set 999 and va...,These values are sentinels or implausible valu...
6,visits,impossible and loosely written visit dates,Keep dates that are only in the YYYY-MM-DD for...,Remove falsely documented dates.
7,visits,repeated patient_id and visit pairs,Sort repeated patient_id and visit pairs by vi...,The repeated patient_id and visit happen when ...
8,sites,repeated site_code rows,Keep the repeated site codes.,The repeated site codes represent records from...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [26]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
display(raw["a1c_wide"])

# 1 Melt the wide A1C data.
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct"
)

# 2 Map the visit labels as the same as visit_clean
a1c_long["visit"] = a1c_long["visit"].map({
    "a1c_baseline": "baseline",
    "a1c_week12": "week12"
})

# 3 Check the type of a1c_pct, and drop the rows with no result.
print(a1c_long["a1c_pct"].dtype)

a1c_long = (
    a1c_long
    .dropna(subset=["a1c_pct"])
    .reset_index(drop=True)
)

# create and save a per-visit summary
a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [
        (a1c_long["visit"] == "baseline").sum(),
        (a1c_long["visit"] == "week12").sum()
    ],
    "a1c_pct_mean": [
        round(
            a1c_long.loc[
                a1c_long["visit"] == "baseline",
                "a1c_pct"
            ].mean(),
            2
        ),
        round(
            a1c_long.loc[
                a1c_long["visit"] == "week12",
                "a1c_pct"
            ].mean(),
            2
        )
    ]
})

display(a1c_long)
display(a1c_summary)

,patient_id,a1c_baseline,a1c_week12
0,P00001,7.3,NaN
1,P00002,8.4,7.4
2,P00003,5.9,5.6
3,P00004,7.3,7.1
4,P00005,7.9,7.6
...,...,...,...
29419,P90026,6.7,6.1
29420,P90027,7.6,7.4
29421,P90028,7.3,7.0
29422,P90029,8.6,NaN


float64


,patient_id,visit,a1c_pct
0,P00001,baseline,7.3
1,P00002,baseline,8.4
2,P00003,baseline,5.9
3,P00004,baseline,7.3
4,P00005,baseline,7.9
...,...,...,...
56755,P90025,week12,7.5
56756,P90026,week12,6.1
56757,P90027,week12,7.4
56758,P90028,week12,7.0


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [13]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [14]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [15]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [17]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [19]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [20]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.